# 19 — Temporal EES trajectory

This consumer notebook traces the population-weighted EES study-area scores from 2025 through 2055 under the real EIA-860 baseline retirement schedule and four player-decision scenarios. It uses only the public lifecycle API in `src/terra_engine.py`; the engine and golden fixtures are not modified.

The simulation uses the default historical climate context (`None`). Per the C0/C3 build notes, historical context is a no-op; non-historical climate lenses are a separate path and are not exercised here. The 2025 row is the initialized baseline. One `advance_year` call is made for each subsequent year, 2026–2055, and the resulting `state['history']` snapshot is checked against `compute_ees_summary` for every annual row.

The front-loaded and paced portfolios use the first five entries of the corrected sourced-capex ranking: `prairie_restoration`, `invasive_treatment`, `irrigation_efficiency`, `riparian_buffer`, and `workforce_retraining`. The succession-aware portfolio carries the same five sourced interventions and adds two site-timed `smr_advanced` successor builds at Dave Johnston and Jim Bridger. Those two successor actions are included for their current SITE_COMPAT behavior but have unsourced capex, so their cost contribution is never folded into sourced totals or normalized comparisons.

In [1]:
from pathlib import Path
import inspect
import json
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import terra_engine as te

RETIREMENT_PATH = PROJECT_ROOT / 'terra-app' / 'src' / 'data' / 'baseline_retirements.json'
RANKING_PATH = DATA_DIR / 'sweep_cost_ranking_sourced.csv'
PORTFOLIO_PATH = DATA_DIR / 'mc_representative_portfolio.json'
with RETIREMENT_PATH.open() as handle:
    BASELINE_RETIREMENTS = json.load(handle)
with (DATA_DIR / 'mw_anchor_facilities.geojson').open() as handle:
    ANCHOR_FACILITIES = json.load(handle)
with (DATA_DIR / 'asset_exposure_tags.json').open() as handle:
    EXPOSURE_TAG_DATA = json.load(handle)
RANKING = pd.read_csv(RANKING_PATH)
with PORTFOLIO_PATH.open() as handle:
    REPRESENTATIVE_PORTFOLIO = json.load(handle)['portfolio']

PUBLIC_API = [
    'initialize_state', 'apply_action', 'advance_year',
    'schedule_retirement', 'accelerate_retirement', 'delay_retirement',
    'cancel_queued', 'queue_action', 'compute_ees_summary',
]
RUNTIME_SIGNATURES = {name: str(inspect.signature(getattr(te, name))) for name in PUBLIC_API}
display(pd.DataFrame({'function': list(RUNTIME_SIGNATURES), 'runtime_signature': list(RUNTIME_SIGNATURES.values())}))
assert te._HAS_INDICATORS, 'state history snapshots are required but indicators.py did not import'
assert te.EMPTY_CLIMATE_CONTEXT == {'lens': 'historical', 'tables': {}}
print(f'Engine module: {te.__file__}')
print(f'Historical climate context: {te.EMPTY_CLIMATE_CONTEXT}')

,function,runtime_signature
0,initialize_state,"(data_dir=None, county_ees_path=None, crosswal..."
1,apply_action,"(state, action_id, location, magnitude, _skip_..."
2,advance_year,"(state, climate_context=None)"
3,schedule_retirement,"(state, asset_id, year)"
4,accelerate_retirement,"(state, asset_id, new_year)"
5,delay_retirement,"(state, asset_id, new_year)"
6,cancel_queued,"(state, asset_id)"
7,queue_action,"(state, action_id, geoid, magnitude, decision_..."
8,compute_ees_summary,(state)


Engine module: /Users/dylanhartman/projects/Energy Modeling/energy-map/src/terra_engine.py
Historical climate context: {'lens': 'historical', 'tables': {}}


## Scenario construction and sourced-cost basis

The selected sourced action mix is deliberately tied to `sweep_cost_ranking_sourced.csv`, not the original marginal-return file. Nominal magnitudes and representative locations come from the existing magnitude-sweep portfolio, so each queued tuple has a compatible engine unit and county.

In [2]:
SOURCED = RANKING[RANKING['cost_status'].eq('sourced')].copy()
SOURCED_IDS = set(SOURCED['action_id'])
RANKED_ACTIONS = list(RANKING['action_id'])
FRONT_LOADED_IDS = ['prairie_restoration'] + [
    action_id for action_id in RANKED_ACTIONS
    if action_id != 'prairie_restoration'
][0:4]
assert FRONT_LOADED_IDS == [
    'prairie_restoration', 'invasive_treatment', 'irrigation_efficiency',
    'riparian_buffer', 'workforce_retraining',
]
assert set(FRONT_LOADED_IDS).issubset(SOURCED_IDS)

portfolio_by_action = {entry['action_id']: entry for entry in REPRESENTATIVE_PORTFOLIO}
assert set(FRONT_LOADED_IDS).issubset(portfolio_by_action)

def sourced_unit_cost(action_id):
    row = SOURCED.loc[SOURCED['action_id'].eq(action_id)]
    if row.empty:
        return None
    value = row.iloc[0]['capex_per_unit']
    return float(value) if pd.notna(value) else None

def action_tuple(action_id, decision_year):
    entry = portfolio_by_action[action_id]
    return (action_id, str(entry['geoid']).zfill(5), float(entry['magnitude']), int(decision_year))

def sourced_capex(action_id, magnitude):
    unit_cost = sourced_unit_cost(action_id)
    return None if unit_cost is None else unit_cost * float(magnitude)

front_years = dict(zip(FRONT_LOADED_IDS, [2026, 2026, 2027, 2027, 2028]))
paced_years = dict(zip(FRONT_LOADED_IDS, [2026, 2032, 2038, 2044, 2044]))
sourced_front = [action_tuple(action_id, front_years[action_id]) for action_id in FRONT_LOADED_IDS]
sourced_paced = [action_tuple(action_id, paced_years[action_id]) for action_id in FRONT_LOADED_IDS]

SCENARIOS = {
    'do_nothing': [],
    'front_loaded': sourced_front,
    'paced': sourced_paced,
    'succession_aware': sourced_paced + [
        ('smr_advanced', '56009', 100.0, 2028),  # Dave Johnston successor
        ('smr_advanced', '56037', 100.0, 2032),  # Jim Bridger successor
    ],
}

def scenario_capex_table():
    rows = []
    for scenario, decisions in SCENARIOS.items():
        sourced_total = 0.0
        unsourced_actions = []
        for action_id, geoid, magnitude, decision_year in decisions:
            cost = sourced_capex(action_id, magnitude)
            if cost is None:
                unsourced_actions.append(action_id)
            else:
                sourced_total += cost
        rows.append({
            'scenario': scenario,
            'sourced_capex_total_usd': sourced_total,
            'unsourced_actions_included': ', '.join(unsourced_actions) or None,
            'unsourced_capex_contribution_usd': np.nan if unsourced_actions else 0.0,
            'cost_basis': '12-action sourced list only; unsourced contribution null' if unsourced_actions else '12-action sourced list only',
        })
    return pd.DataFrame(rows)

display(SOURCED[['action_id', 'best_magnitude_pct', 'capex_per_unit', 'capex_source']].head(12))
display(pd.DataFrame([
    {'scenario': name, 'queued_tuples': decisions}
    for name, decisions in SCENARIOS.items()
]))
display(scenario_capex_table())

,action_id,best_magnitude_pct,capex_per_unit,capex_source
0,prairie_restoration,25.0,150.0,USDA EQIP Practice 643 (Native Pasture and Ran...
1,invasive_treatment,100.0,80.0,USDA EQIP Practice 315 (Herbaceous Weed Contro...
2,irrigation_efficiency,75.0,600.0,"USDA EQIP Practice 441 (Irrigation System, Mic..."
3,riparian_buffer,75.0,8000.0,USDA EQIP Practice 391 (Riparian Forest Buffer...
4,workforce_retraining,25.0,8500.0,DOL Workforce Innovation and Opportunity Act p...
5,transmission_230kv,75.0,2500000.0,"Liming & Tegen (2011, NREL TP-5500-48175); DOE..."
6,affordable_housing,25.0,180000.0,NLIHC Out of Reach 2023; HUD affordable housin...
7,health_clinic,25.0,4500000.0,HRSA FQHC New Access Points capital cost estim...
8,rural_broadband,50.0,3500.0,FCC Rural Digital Opportunity Fund program dat...
9,coal_repowering,50.0,300000.0,NETL Cost and Performance Baseline for Fossil ...


,scenario,queued_tuples
0,do_nothing,[]
1,front_loaded,"[(prairie_restoration, 08001, 10000.0, 2026), ..."
2,paced,"[(prairie_restoration, 08001, 10000.0, 2026), ..."
3,succession_aware,"[(prairie_restoration, 08001, 10000.0, 2026), ..."


,scenario,sourced_capex_total_usd,unsourced_actions_included,unsourced_capex_contribution_usd,cost_basis
0,do_nothing,0.0,NaN,0.0,12-action sourced list only
1,front_loaded,91400000.0,NaN,0.0,12-action sourced list only
2,paced,91400000.0,NaN,0.0,12-action sourced list only
3,succession_aware,91400000.0,"smr_advanced, smr_advanced",NaN,12-action sourced list only; unsourced contrib...


## Run the annual lifecycle trajectories

Actions are queued when the simulation reaches their decision year. This ordering is required for the succession case: the 2027 and 2031 retirements must fire first so their sites exist when the 2028 and 2032 successor decisions are queued.

In [3]:
YEARS = list(range(2025, 2056))
CAPITALS = ['E', 'Ec', 'S']
SCENARIO_LABELS = {
    'do_nothing': 'Do-nothing',
    'front_loaded': 'Front-loaded build',
    'paced': 'Paced build',
    'succession_aware': 'Site-succession-aware build',
}

def retirements_fired_between(before_state, after_state):
    before = {asset['asset_id']: asset.get('lifecycle') for asset in before_state['asset_registry']}
    fired = []
    for asset in after_state['asset_registry']:
        if (asset.get('lifecycle') == 'retired'
                and before.get(asset['asset_id']) != 'retired'
                and asset.get('scheduled_retirement_year') == after_state['year']):
            fired.append(asset)
    return fired

def study_scores_from_state(state):
    summary = te.compute_ees_summary(state)
    summary_study = summary['study_area']
    history = state.get('history', [])
    if state['year'] == 2025:
        assert not history, 'the initialized 2025 state should not yet have a history snapshot'
    else:
        assert history and history[-1]['year'] == state['year']
        history_study = history[-1]['study']
        for capital in CAPITALS:
            assert abs(float(history_study[capital]) - float(summary_study[capital])) < 0.0001, (
                state['year'], capital, history_study[capital], summary_study[capital]
            )
    return {capital: float(summary_study[capital]) for capital in CAPITALS}

def run_scenario(scenario_name, decisions, baseline_retirements=BASELINE_RETIREMENTS):
    state = te.initialize_state(
        data_dir=DATA_DIR,
        baseline_retirements=baseline_retirements,
        start_year=2025,
        anchor_facilities=ANCHOR_FACILITIES,
        exposure_tag_data=EXPOSURE_TAG_DATA,
    )
    decisions_by_year = {}
    for decision in decisions:
        decisions_by_year.setdefault(int(decision[3]), []).append(decision)
    cumulative_sourced = 0.0
    unsourced_included = False
    rows = []
    queue_log = []

    for year in YEARS:
        active_retirements_ytd = 0
        if year > 2025:
            before_state = state
            state = te.advance_year(state, climate_context=None)
            fired = retirements_fired_between(before_state, state)
            active_retirements_ytd = len(fired)

        for action_id, geoid, magnitude, decision_year in decisions_by_year.get(year, []):
            state = te.queue_action(
                state, action_id, geoid, magnitude, decision_year,
                climate_context=None,
            )
            cost = sourced_capex(action_id, magnitude)
            if cost is None:
                unsourced_included = True
            else:
                cumulative_sourced += cost
            queue_log.append({
                'decision_year': year, 'action_id': action_id, 'geoid': geoid,
                'magnitude': magnitude, 'sourced_capex_usd': cost,
            })

        scores = study_scores_from_state(state)
        rows.append({
            'scenario': scenario_name,
            'year': year,
            **scores,
            'composite': float(np.mean([scores[capital] for capital in CAPITALS])),
            'active_retirements_ytd': active_retirements_ytd,
            'cumulative_capex_sourced': cumulative_sourced,
            'cumulative_capex_unsourced_actions_included': np.nan if unsourced_included else 0.0,
        })

    return {
        'rows': pd.DataFrame(rows),
        'final_state': state,
        'queue_log': pd.DataFrame(queue_log),
    }

RUNS = {name: run_scenario(name, decisions) for name, decisions in SCENARIOS.items()}
TRAJECTORY = pd.concat([run['rows'] for run in RUNS.values()], ignore_index=True)
TRAJECTORY = TRAJECTORY[[
    'scenario', 'year', 'E', 'Ec', 'S', 'composite',
    'active_retirements_ytd', 'cumulative_capex_sourced',
    'cumulative_capex_unsourced_actions_included',
]]
assert len(TRAJECTORY) == 4 * len(YEARS)
assert TRAJECTORY.groupby('scenario')['year'].nunique().eq(len(YEARS)).all()
TRAJECTORY_PATH = DATA_DIR / 'trajectory_results.csv'
TRAJECTORY.to_csv(TRAJECTORY_PATH, index=False)
print(f'Saved {TRAJECTORY_PATH} with {len(TRAJECTORY)} rows')
display(TRAJECTORY.head(8))

Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/trajectory_results.csv with 124 rows


,scenario,year,E,Ec,S,composite,active_retirements_ytd,cumulative_capex_sourced,cumulative_capex_unsourced_actions_included
0,do_nothing,2025,3.1098,6.9322,5.2827,5.108233,0,0.0,0.0
1,do_nothing,2026,3.1083,6.9343,5.2834,5.108667,0,0.0,0.0
2,do_nothing,2027,3.1069,6.9336,5.2840,5.108167,1,0.0,0.0
3,do_nothing,2028,3.1055,6.9358,5.2847,5.108667,0,0.0,0.0
4,do_nothing,2029,3.1040,6.9379,5.2853,5.109067,0,0.0,0.0
5,do_nothing,2030,3.1026,6.9400,5.2860,5.109533,0,0.0,0.0
6,do_nothing,2031,3.1012,6.9392,5.2866,5.109000,1,0.0,0.0
7,do_nothing,2032,3.0997,6.9414,5.2872,5.109433,0,0.0,0.0


In [4]:
# Confirm that the succession case actually exercised the live SITE_COMPAT fields.
succession_assets = [
    asset for asset in RUNS['succession_aware']['final_state']['asset_registry']
    if asset.get('origin') == 'player' and asset.get('action_id') == 'smr_advanced'
]
succession_audit = pd.DataFrame({
    'action_id': asset['action_id'], 'geoid': asset['geoid'],
    'decision_year': asset['decision_year'], 'operational_year': asset['operational_year'],
    'succession_site_id': asset['succession_site_id'],
    'ttd_reduction_applied': asset['ttd_reduction_applied'],
    'capex_discount_fraction': asset['capex_discount_fraction'],
    'tx_waiver_mw': asset['tx_waiver_mw'],
} for asset in succession_assets)
assert len(succession_audit) == 2
assert succession_audit['ttd_reduction_applied'].eq(2).all()
assert succession_audit['capex_discount_fraction'].eq(0.15).all()
display(succession_audit)
print('The current engine gives coal_to_smr a TX waiver only (no TTD/capex discount) and does not list coal_to_solar as a thermal SITE_COMPAT action; smr_advanced is the compatible equivalent successor used here.')

,action_id,geoid,decision_year,operational_year,succession_site_id,ttd_reduction_applied,capex_discount_fraction,tx_waiver_mw
0,smr_advanced,56009,2028,2038,site_56009_dave_johnston_power_plant_2027,2,0.15,100.0
1,smr_advanced,56037,2032,2042,site_56037_jim_bridger_power_plant_2031,2,0.15,100.0


The current engine gives coal_to_smr a TX waiver only (no TTD/capex discount) and does not list coal_to_solar as a thermal SITE_COMPAT action; smr_advanced is the compatible equivalent successor used here.


In [5]:
RETIREMENT_YEARS = [2027, 2031]
do_nothing = RUNS['do_nothing']['rows'].set_index('year')
retirement_rows = []
for year in RETIREMENT_YEARS:
    prior = do_nothing.loc[year - 1]
    current = do_nothing.loc[year]
    retirement_rows.append({
        'year': year,
        'active_retirements_ytd': int(current['active_retirements_ytd']),
        'delta_E_yoy': current['E'] - prior['E'],
        'delta_Ec_yoy': current['Ec'] - prior['Ec'],
        'delta_S_yoy': current['S'] - prior['S'],
        'delta_composite_yoy': current['composite'] - prior['composite'],
    })
RETIREMENT_IMPACT = pd.DataFrame(retirement_rows)
display(RETIREMENT_IMPACT.style.format({column: '{:+.6f}' for column in RETIREMENT_IMPACT.columns if column.startswith('delta_')}))
assert RETIREMENT_IMPACT['active_retirements_ytd'].tolist() == [1, 1]

,year,active_retirements_ytd,delta_E_yoy,delta_Ec_yoy,delta_S_yoy,delta_composite_yoy
0,2027,1,-0.001400,-0.000700,+0.000600,-0.000500
1,2031,1,-0.001400,-0.000800,+0.000600,-0.000533


In [6]:
# Isolate the scheduled-retirement contribution from autonomous drift.
no_retirement_control = run_scenario('no_retirement_control', [])['rows'].set_index('year')
single_retirement_controls = {}
for geoid, plant_name in [('56009', 'Dave Johnston Power Plant'), ('56037', 'Jim Bridger Power Plant')]:
    filtered = {geoid: {plant_name: BASELINE_RETIREMENTS[geoid][plant_name]}}
    single_retirement_controls[geoid] = run_scenario(
        f'single_{geoid}', [], baseline_retirements=filtered
    )['rows'].set_index('year')

for row in retirement_rows:
    year = row['year']
    geoid = '56009' if year == 2027 else '56037'
    single = single_retirement_controls[geoid].loc[year]
    control = no_retirement_control.loc[year]
    row['isolated_retirement_effect_E'] = single['E'] - control['E']
    row['isolated_retirement_effect_Ec'] = single['Ec'] - control['Ec']
    row['isolated_retirement_effect_S'] = single['S'] - control['S']
    row['isolated_retirement_effect_composite'] = single['composite'] - control['composite']
RETIREMENT_IMPACT = pd.DataFrame(retirement_rows)
display(RETIREMENT_IMPACT.style.format({column: '{:+.6f}' for column in RETIREMENT_IMPACT.columns if 'delta_' in column or 'effect_' in column}))

,year,active_retirements_ytd,delta_E_yoy,delta_Ec_yoy,delta_S_yoy,delta_composite_yoy,isolated_retirement_effect_E,isolated_retirement_effect_Ec,isolated_retirement_effect_S,isolated_retirement_effect_composite
0,2027,1,-0.001400,-0.000700,+0.000600,-0.000500,+0.000000,+0.000000,+0.000000,+0.000000
1,2031,1,-0.001400,-0.000800,+0.000600,-0.000533,+0.000000,+0.002800,+0.000000,+0.000933


## Trajectory plots

Vertical markers show the scheduled baseline retirements: Dave Johnston in 2027 and Jim Bridger in 2031.

In [7]:
COLORS = {
    'do_nothing': '#A9B3BF',
    'front_loaded': '#2DD4BF',
    'paced': '#F59E0B',
    'succession_aware': '#A78BFA',
}

def save_trajectory_plot(metric, ylabel, filename, title):
    fig, ax = plt.subplots(figsize=(11, 6.2), dpi=160)
    for scenario in SCENARIOS:
        frame = TRAJECTORY[TRAJECTORY['scenario'].eq(scenario)]
        ax.plot(frame['year'], frame[metric], label=SCENARIO_LABELS[scenario],
                color=COLORS[scenario], linewidth=2.2 if scenario != 'do_nothing' else 2.6)
    ax.axvline(2027, color='#EF4444', linestyle='--', linewidth=1.2, alpha=0.75, label='Dave Johnston retirement (2027)')
    ax.axvline(2031, color='#F97316', linestyle=':', linewidth=1.5, alpha=0.85, label='Jim Bridger retirement (2031)')
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel(ylabel)
    ax.set_xlim(2025, 2055)
    ax.grid(alpha=0.25)
    ax.legend(loc='best', fontsize=8, frameon=False)
    fig.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    print(f'Saved {path}')

save_trajectory_plot('composite', 'Composite EES score (mean of E, Ec, S)', 'trajectory_composite.png', 'TERRA temporal trajectory — composite EES score')
save_trajectory_plot('E', 'E score', 'trajectory_E.png', 'TERRA temporal trajectory — Environmental capital (E)')
save_trajectory_plot('Ec', 'Ec score', 'trajectory_Ec.png', 'TERRA temporal trajectory — Economic capital (Ec)')
save_trajectory_plot('S', 'S score', 'trajectory_S.png', 'TERRA temporal trajectory — Social capital (S)')

<Figure size 1760x992 with 1 Axes>

Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/figures/trajectory_composite.png


<Figure size 1760x992 with 1 Axes>

Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/figures/trajectory_E.png


<Figure size 1760x992 with 1 Axes>

Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/figures/trajectory_Ec.png


<Figure size 1760x992 with 1 Axes>

Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/figures/trajectory_S.png


In [8]:
# Succession value at matched sourced capex. Scenario 4 carries the same priced
# source mix as paced and adds two unpriced successor actions.
CHECKPOINTS = [2027, 2031, 2038, 2044, 2050, 2055]
paced = RUNS['paced']['rows'].set_index('year')
succession = RUNS['succession_aware']['rows'].set_index('year')
succession_comparison = pd.DataFrame([{
    'year': year,
    'paced_composite': paced.loc[year, 'composite'],
    'succession_aware_composite': succession.loc[year, 'composite'],
    'delta_including_unsourced_successors': succession.loc[year, 'composite'] - paced.loc[year, 'composite'],
    'paced_sourced_capex_usd': paced.loc[year, 'cumulative_capex_sourced'],
    'succession_sourced_capex_usd': succession.loc[year, 'cumulative_capex_sourced'],
    'unsourced_capex_contribution_usd': np.nan,
    'delta_at_matched_sourced_capex': succession.loc[year, 'composite'] - paced.loc[year, 'composite'],
}] for year in CHECKPOINTS)
display(succession_comparison.style.format({
    'paced_composite': '{:.6f}', 'succession_aware_composite': '{:.6f}',
    'delta_including_unsourced_successors': '{:+.6f}',
    'paced_sourced_capex_usd': '${:,.0f}', 'succession_sourced_capex_usd': '${:,.0f}',
    'delta_at_matched_sourced_capex': '{:+.6f}',
}))
print('All sourced-capex totals match between paced and succession-aware builds; the two successor actions remain unpriced (null), so no unsourced dollar amount is included.')

,0
0,"{'year': 2027, 'paced_composite': np.float64(5.144066666666666), 'succession_aware_composite': np.float64(5.144066666666666), 'delta_including_unsourced_successors': np.float64(0.0), 'paced_sourced_capex_usd': np.float64(1500000.0), 'succession_sourced_capex_usd': np.float64(1500000.0), 'unsourced_capex_contribution_usd': nan, 'delta_at_matched_sourced_capex': np.float64(0.0)}"
1,"{'year': 2031, 'paced_composite': np.float64(5.144933333333333), 'succession_aware_composite': np.float64(5.144933333333333), 'delta_including_unsourced_successors': np.float64(0.0), 'paced_sourced_capex_usd': np.float64(1500000.0), 'succession_sourced_capex_usd': np.float64(1500000.0), 'unsourced_capex_contribution_usd': nan, 'delta_at_matched_sourced_capex': np.float64(0.0)}"
2,"{'year': 2038, 'paced_composite': np.float64(5.149133333333334), 'succession_aware_composite': np.float64(5.1494), 'delta_including_unsourced_successors': np.float64(0.0002666666666657491), 'paced_sourced_capex_usd': np.float64(2900000.0), 'succession_sourced_capex_usd': np.float64(2900000.0), 'unsourced_capex_contribution_usd': nan, 'delta_at_matched_sourced_capex': np.float64(0.0002666666666657491)}"
3,"{'year': 2044, 'paced_composite': np.float64(5.152233333333333), 'succession_aware_composite': np.float64(5.1531666666666665), 'delta_including_unsourced_successors': np.float64(0.0009333333333332305), 'paced_sourced_capex_usd': np.float64(91400000.0), 'succession_sourced_capex_usd': np.float64(91400000.0), 'unsourced_capex_contribution_usd': nan, 'delta_at_matched_sourced_capex': np.float64(0.0009333333333332305)}"
4,"{'year': 2050, 'paced_composite': np.float64(5.188), 'succession_aware_composite': np.float64(5.1887), 'delta_including_unsourced_successors': np.float64(0.000700000000000145), 'paced_sourced_capex_usd': np.float64(91400000.0), 'succession_sourced_capex_usd': np.float64(91400000.0), 'unsourced_capex_contribution_usd': nan, 'delta_at_matched_sourced_capex': np.float64(0.000700000000000145)}"
5,"{'year': 2055, 'paced_composite': np.float64(5.189933333333333), 'succession_aware_composite': np.float64(5.190433333333334), 'delta_including_unsourced_successors': np.float64(0.0005000000000006111), 'paced_sourced_capex_usd': np.float64(91400000.0), 'succession_sourced_capex_usd': np.float64(91400000.0), 'unsourced_capex_contribution_usd': nan, 'delta_at_matched_sourced_capex': np.float64(0.0005000000000006111)}"


All sourced-capex totals match between paced and succession-aware builds; the two successor actions remain unpriced (null), so no unsourced dollar amount is included.


In [9]:
# Plain-language handoff summary.
baseline_2025 = do_nothing.loc[2025, 'composite']
baseline_2055 = do_nothing.loc[2055, 'composite']
no_retire_2055 = no_retirement_control.loc[2055, 'composite']
total_change = baseline_2055 - baseline_2025
retirement_component = baseline_2055 - no_retire_2055
drift_component = no_retire_2055 - baseline_2025
sourced_total = scenario_capex_table().set_index('scenario').loc['paced', 'sourced_capex_total_usd']
unsourced_count = len([row for row in SCENARIOS['succession_aware'] if sourced_capex(row[0], row[2]) is None])
delta_2055 = succession.loc[2055, 'composite'] - paced.loc[2055, 'composite']
summary_text = f'''
### Plain-language summary

In the do-nothing trajectory, the composite score changes from **{baseline_2025:.4f}** in 2025 to **{baseline_2055:.4f}** in 2055, a net change of **{total_change:+.4f}**. Using a matched no-scheduled-retirement control, the 2055 change decomposes into approximately **{retirement_component:+.4f}** attributable to the scheduled retirement path and **{drift_component:+.4f}** from autonomous drift and other modeled dynamics. The direct year-over-year retirement impacts are shown in the table above; those deltas include the full annual lifecycle step, while the isolated columns compare one scheduled retirement against the no-retirement control.

The succession-aware build is held to the paced build's same sourced intervention mix and sourced capex total (**${sourced_total:,.0f}**). At 2055 its composite-score difference is **{delta_2055:+.4f}**; checkpoint values above show whether that advantage appears early or only after commissioning. The two thermal-site `smr_advanced` successors exercise the current low-confidence SITE_COMPAT discount and are included in the score trajectory, but their capex is unsourced and therefore **{unsourced_count} action contributions are null for cost normalization**. The matched-sourced-capex comparison is thus directional on score effects and does not claim a $/point result for those successors.

The solid cost ground is limited to the **12 of 30 actions** with usable sourced capex in `sweep_cost_ranking_sourced.csv`; this notebook uses five of those twelve for the front-loaded and paced portfolios. The succession-aware scenario's sourced total is defensible, but its added site-successor costs are not priced. The comparison is therefore cost-grounded for the common five-action portfolio and directional-only for the unpriced succession additions.

Climate choice: `climate_context=None`, which resolves to the historical lens. Non-historical climate lenses were intentionally not exercised.
'''
display(Markdown(summary_text))


### Plain-language summary

In the do-nothing trajectory, the composite score changes from **5.1082** in 2025 to **5.1187** in 2055, a net change of **+0.0104**. Using a matched no-scheduled-retirement control, the 2055 change decomposes into approximately **+0.0000** attributable to the scheduled retirement path and **+0.0104** from autonomous drift and other modeled dynamics. The direct year-over-year retirement impacts are shown in the table above; those deltas include the full annual lifecycle step, while the isolated columns compare one scheduled retirement against the no-retirement control.

The succession-aware build is held to the paced build's same sourced intervention mix and sourced capex total (**$91,400,000**). At 2055 its composite-score difference is **+0.0005**; checkpoint values above show whether that advantage appears early or only after commissioning. The two thermal-site `smr_advanced` successors exercise the current low-confidence SITE_COMPAT discount and are included in the score trajectory, but their capex is unsourced and therefore **2 action contributions are null for cost normalization**. The matched-sourced-capex comparison is thus directional on score effects and does not claim a $/point result for those successors.

The solid cost ground is limited to the **12 of 30 actions** with usable sourced capex in `sweep_cost_ranking_sourced.csv`; this notebook uses five of those twelve for the front-loaded and paced portfolios. The succession-aware scenario's sourced total is defensible, but its added site-successor costs are not priced. The comparison is therefore cost-grounded for the common five-action portfolio and directional-only for the unpriced succession additions.

Climate choice: `climate_context=None`, which resolves to the historical lens. Non-historical climate lenses were intentionally not exercised.


## Diagnostic addendum: retirement direction, raw precision, and succession timing

This addendum re-derives the study-area E/Ec/S population-weighted means directly from the unrounded county state. It writes raw retirement-only and population-drift Ec columns back to `trajectory_results.csv`. Each retirement year is compared with a control that preserves all prior retirements but omits the event firing in that year.

In [10]:
def raw_study_scores(state):
    county_rows = list(state['county_ees'].values())
    population_total = sum(float(row.get('population', 1.0)) for row in county_rows)
    scores = {
        capital: sum(float(row[capital]) * float(row.get('population', 1.0)) for row in county_rows) / population_total
        for capital in CAPITALS
    }
    scores['composite'] = sum(scores[capital] for capital in CAPITALS) / len(CAPITALS)
    return scores

def raw_trajectory(decisions, baseline_retirements=BASELINE_RETIREMENTS):
    state = te.initialize_state(
        data_dir=DATA_DIR, baseline_retirements=baseline_retirements, start_year=2025,
        anchor_facilities=ANCHOR_FACILITIES, exposure_tag_data=EXPOSURE_TAG_DATA,
    )
    decisions_by_year = {}
    for action_id, location, magnitude, decision_year in decisions:
        decisions_by_year.setdefault(int(decision_year), []).append((action_id, location, magnitude, decision_year))
    rows = {2025: raw_study_scores(state)}
    for year in range(2026, 2056):
        state = te.advance_year(state, climate_context=None)
        for action_id, location, magnitude, decision_year in decisions_by_year.get(year, []):
            state = te.queue_action(state, action_id, location, magnitude, decision_year, climate_context=None)
        rows[year] = raw_study_scores(state)
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('year').sort_index()

raw_do_nothing = raw_trajectory([])
raw_no_retirement = raw_trajectory([], baseline_retirements=[])
dave_only_retirements = {
    '56009': {'Dave Johnston Power Plant': BASELINE_RETIREMENTS['56009']['Dave Johnston Power Plant']}
}
raw_dave_only = raw_trajectory([], baseline_retirements=dave_only_retirements)
retirement_controls = {2027: raw_no_retirement, 2031: raw_dave_only}
retirement_metadata = {
    2027: {'plant_code': '4158', 'facility': 'Dave Johnston Power Plant', 'county_geoid': '56009', 'county': 'Converse'},
    2031: {'plant_code': '8066', 'facility': 'Jim Bridger Power Plant', 'county_geoid': '56037', 'county': 'Sweetwater'},
}
retirement_rows_raw = []
for year, control in retirement_controls.items():
    raw_yoy_delta_Ec = raw_do_nothing.loc[year, 'Ec'] - raw_do_nothing.loc[year - 1, 'Ec']
    retirement_only_delta_Ec = raw_do_nothing.loc[year, 'Ec'] - control.loc[year, 'Ec']
    population_drift_delta_Ec = control.loc[year, 'Ec'] - control.loc[year - 1, 'Ec']
    meta = retirement_metadata[year]
    retirement_rows_raw.append({
        'year': year,
        **meta,
        'raw_yoy_delta_E': raw_do_nothing.loc[year, 'E'] - raw_do_nothing.loc[year - 1, 'E'],
        'raw_yoy_delta_Ec': raw_yoy_delta_Ec,
        'raw_yoy_delta_S': raw_do_nothing.loc[year, 'S'] - raw_do_nothing.loc[year - 1, 'S'],
        'raw_yoy_delta_composite': raw_do_nothing.loc[year, 'composite'] - raw_do_nothing.loc[year - 1, 'composite'],
        'retirement_only_delta_Ec': retirement_only_delta_Ec,
        'population_drift_delta_Ec': population_drift_delta_Ec,
        'decomposition_error_Ec': raw_yoy_delta_Ec - retirement_only_delta_Ec - population_drift_delta_Ec,
    })
raw_retirement_impact = pd.DataFrame(retirement_rows_raw)
assert raw_retirement_impact['decomposition_error_Ec'].abs().max() < 1e-12
for column in ['retirement_only_delta_Ec', 'population_drift_delta_Ec']:
    TRAJECTORY[column] = np.nan
for column in ['retirement_plant_code', 'retirement_facility', 'retirement_county_geoid', 'retirement_county']:
    TRAJECTORY[column] = None
for row in raw_retirement_impact.itertuples():
    mask = TRAJECTORY['scenario'].eq('do_nothing') & TRAJECTORY['year'].eq(row.year)
    TRAJECTORY.loc[mask, 'retirement_only_delta_Ec'] = row.retirement_only_delta_Ec
    TRAJECTORY.loc[mask, 'population_drift_delta_Ec'] = row.population_drift_delta_Ec
    TRAJECTORY.loc[mask, 'retirement_plant_code'] = row.plant_code
    TRAJECTORY.loc[mask, 'retirement_facility'] = row.facility
    TRAJECTORY.loc[mask, 'retirement_county_geoid'] = row.county_geoid
    TRAJECTORY.loc[mask, 'retirement_county'] = row.county
TRAJECTORY.to_csv(TRAJECTORY_PATH, index=False)
display(raw_retirement_impact.style.format({
    'raw_yoy_delta_E': '{:+.12f}', 'raw_yoy_delta_Ec': '{:+.12f}',
    'raw_yoy_delta_S': '{:+.12f}', 'raw_yoy_delta_composite': '{:+.12f}',
    'retirement_only_delta_Ec': '{:+.12f}', 'population_drift_delta_Ec': '{:+.12f}',
    'decomposition_error_Ec': '{:+.3e}',
}))

raw_paced = raw_trajectory(SCENARIOS['paced'])
raw_succession = raw_trajectory(SCENARIOS['succession_aware'])
raw_succession_delta = pd.DataFrame({
    f'delta_{capital}': raw_succession[capital] - raw_paced[capital]
    for capital in [*CAPITALS, 'composite']
})
raw_succession_delta = raw_succession_delta.loc[2026:2055].reset_index()
display(raw_succession_delta.style.format({
    column: '{:+.12f}' for column in raw_succession_delta.columns if column != 'year'
}))

raw_2055_delta = float(raw_succession_delta.loc[raw_succession_delta['year'].eq(2055), 'delta_composite'].iloc[0])
raw_peak_row = raw_succession_delta.loc[raw_succession_delta['delta_composite'].idxmax()]
ret2027 = raw_retirement_impact.set_index('year').loc[2027]
ret2031 = raw_retirement_impact.set_index('year').loc[2031]
display(Markdown(f'''
### Findings

The raw 2027 Ec change separates into **{ret2027['retirement_only_delta_Ec']:+.12f}** from retiring EIA plant 4158 (Dave Johnston, Converse; with its modeled 50 km contribution also reaching Natrona) and **{ret2027['population_drift_delta_Ec']:+.12f}** from the no-new-retirement control's population drift.

The raw 2031 Ec change separates into **{ret2031['retirement_only_delta_Ec']:+.12f}** from corrected EIA plant **8066**, Jim Bridger in **Sweetwater County (56037)**, and **{ret2031['population_drift_delta_Ec']:+.12f}** from population drift with the prior Dave Johnston retirement preserved. The decomposition residual is below 1e-12 in both years. This 2031 result is meaningfully different from the earlier zero-effect finding: it is attached to the corrected physical plant and county, not a refinement of any prior Platte/Laramie River result.

The raw succession-aware minus paced composite delta is **{raw_2055_delta:+.12f}** at 2055. It first appears when the successor actions commission, remains positive thereafter, and peaks at **{raw_peak_row['delta_composite']:+.12f}** in **{int(raw_peak_row['year'])}** before declining to 2055. That is a deterministic modeled signal, but it is neither monotonic nor a steadily growing timing advantage. Given the current low-confidence SITE_COMPAT succession parameters and the small size relative to the engine's precision, treat succession timing as a provisional directional result, not an optimizer-grade finding.
'''))

,year,plant_code,facility,county_geoid,county,raw_yoy_delta_E,raw_yoy_delta_Ec,raw_yoy_delta_S,raw_yoy_delta_composite,retirement_only_delta_Ec,population_drift_delta_Ec,decomposition_error_Ec
0,2027,4158,Dave Johnston Power Plant,56009,Converse,-0.001427932242,-0.000665571678,+0.000646300301,-0.000482401206,-0.002788460225,+0.002122888548,+0.000e+00
1,2031,8066,Jim Bridger Power Plant,56037,Sweetwater,-0.001444919849,-0.000788630087,+0.000637683416,-0.000531955507,-0.002906898132,+0.002118268044,+0.000e+00


,year,delta_E,delta_Ec,delta_S,delta_composite
0,2026,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
1,2027,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
2,2028,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
3,2029,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
4,2030,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
5,2031,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
6,2032,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
7,2033,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
8,2034,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000
9,2035,+0.000000000000,+0.000000000000,+0.000000000000,+0.000000000000



### Findings

The raw 2027 Ec change separates into **-0.002788460225** from retiring EIA plant 4158 (Dave Johnston, Converse; with its modeled 50 km contribution also reaching Natrona) and **+0.002122888548** from the no-new-retirement control's population drift.

The raw 2031 Ec change separates into **-0.002906898132** from corrected EIA plant **8066**, Jim Bridger in **Sweetwater County (56037)**, and **+0.002118268044** from population drift with the prior Dave Johnston retirement preserved. The decomposition residual is below 1e-12 in both years. This 2031 result is meaningfully different from the earlier zero-effect finding: it is attached to the corrected physical plant and county, not a refinement of any prior Platte/Laramie River result.

The raw succession-aware minus paced composite delta is **+0.000460196467** at 2055. It first appears when the successor actions commission, remains positive thereafter, and peaks at **+0.001033613669** in **2042** before declining to 2055. That is a deterministic modeled signal, but it is neither monotonic nor a steadily growing timing advantage. Given the current low-confidence SITE_COMPAT succession parameters and the small size relative to the engine's precision, treat succession timing as a provisional directional result, not an optimizer-grade finding.
